In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ.setdefault('AWS_DEFAULT_REGION', 'us-west-2')
_os.environ.setdefault('AWS_REGION', 'us-west-2')
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            # Honor sessions that were handed explicit credentials
            # (e.g. notebooks that assume_role into other accounts):
            # clobbering them would silently run every cross-account
            # client as the ambient identity. Bare/SDK-created sessions
            # (no explicit creds) still get the refreshable shared-file
            # creds so long-running waits survive credential rotation.
            _ex = getattr(self, '_credentials', None)
            if _ex is not None:
                return _ex
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


## SFTTrainer Example - Finetuning with Sagemaker

This notebook demonstrates basic user flow for SFT Finetuning from a model available in Sagemaker Jumpstart.
Information on available models on jumpstart: https://docs.aws.amazon.com/sagemaker/latest/dg/jumpstart-foundation-models-latest.html

### Setup and Configuration

Initialize the environment by importing necessary libraries and configuring AWS credentials

In [2]:
# Configure AWS credentials and region
#! ada credentials update --provider=isengard --account=<> --role=Admin --profile=default --once
#! aws configure set region us-west-2

In [3]:
from sagemaker.train.sft_trainer import SFTTrainer
from sagemaker.train.common import TrainingType
from sagemaker.core.training.configs import InputData
from rich import print as rprint
from rich.pretty import pprint
from sagemaker.core.resources import ModelPackage

import boto3
from sagemaker.core.helper.session_helper import Session
import os


# For MLFlow native metrics in Trainer wait, run below line with approriate region
os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = "https://mlflow.sagemaker.us-west-2.app.aws"



sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


## Finetuning with Jumpstart base model

### Prepare and Register Dataset
Prepare and Register Dataset for Finetuning

In [4]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.dataset_utils import CustomizationTechnique


# Register dataset in SageMaker AI Registry
# This creates a versioned dataset that can be referenced by ARN
# Provide a source: a local file path or your own S3 URI pointing to a JSONL file.
# The dataset must be in JSONL format with each line containing a JSON object
# with 'prompt' and 'completion' fields. See the SageMaker documentation for details:
# https://docs.aws.amazon.com/sagemaker/latest/dg/model-customize-sft.html
#
# Replace the placeholder below with your own S3 URI or local file path:
MY_DATASET_S3_URI = "s3://notebook-test-engine-ds-674622101542-usw2/datasets/sample-sft-dataset.jsonl"  # TODO: replace with your dataset URI

dataset = DataSet.create(
    name="demo-1",
    source=MY_DATASET_S3_URI
)

print(f"Dataset ARN: {dataset.arn}")

[07/21/26 00:21:12] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=4892681;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=4892682;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/21/26 00:21:13] INFO     Cannot simulate policies for                                  ]8;id=4892689;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=4892690;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=4892696;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=4892697;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     Role not provided. Using validated caller role:                         ]8;id=4892704;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=4892705;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#90\90]8;;\
                             arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole                   

[07/21/26 00:21:14] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=4892710;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=4892711;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

...

[07/21/26 00:21:17] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=4892734;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=4892735;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

/usr/local/lib/python3.12/dist-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

[07/21/26 00:21:18] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=4892740;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=4892741;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

Final Resource Status: Available

Dataset ARN: arn:aws:sagemaker:us-west-2:674622101542:hub-content/R8VBAD7SJ9M4M5ATDE0MIPSEJU3M1TEH5UCUNB65JMGNEP87RHFG/DataSet/demo-1/8.0.0


##### Create a Model Package group (if not already exists)

In [5]:
from sagemaker.core.resources import ModelPackage, ModelPackageGroup

model_package_group=ModelPackageGroup.create(model_package_group_name="test-model-package-group")

                    INFO     Creating model_package_group resource.                              ]8;id=4892748;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4892749;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#23250\23250]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=4892756;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=4892757;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#361\361]8;;\

                    INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=4892763;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=4892764;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#375\375]8;;\

### Create SFTTrainer
**Required Parameters** 

* `model`: base_model id on Sagemaker Hubcontent that is available to finetune (or) ModelPackage artifacts

**Optional Parameters**
* `training_type`: Choose from TrainingType Enum(sagemaker.modules.train.common) either LORA OR FULL.
* `model_package_group`: ModelPackage group name or ModelPackageGroup object. This parameter is mandatory when a base model ID is provided, but optional when a model package is provided.
* `mlflow_resource_arn`: MLFlow app ARN to track the training job
* `mlflow_experiment_name`: MLFlow app experiment name(str)
* `mlflow_run_name`: MLFlow app run name(str)
* `training_dataset`: Training Dataset - should be a Dataset ARN or Dataset object (Please note training dataset is required for a training job to run, can be either provided via Trainer or .train())
* `validation_dataset`: Validation Dataset - should be a Dataset ARN or Dataset object
* `s3_output_path`: S3 path for the trained model artifacts

#### Reference 
Refer this doc for other models that support Model Customization: 
https://docs.aws.amazon.com/sagemaker/latest/dg/model-customize-open-weight.html

In [6]:
# For fine-tuning 
sft_trainer = SFTTrainer(
    model="meta-textgeneration-llama-3-2-1b-instruct", 
    training_type=TrainingType.LORA, 
    model_package_group=model_package_group, # or use an existing model package group arn
    mlflow_experiment_name="test-finetuned-models-exp", 
    mlflow_run_name="test-finetuned-models-run", 
    training_dataset=dataset.arn, 
    s3_output_path="s3://notebook-test-engine-ds-674622101542-usw2/output/",  # TODO: replace with your S3 output path
    accept_eula=True
)


[07/21/26 00:21:19] INFO     SageMaker session not provided. Using default Session.                  ]8;id=4892770;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=4892771;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

### Discover and update Finetuning options

Each of the technique and model has overridable hyperparameters that can be finetuned by the user.

In [7]:
print("Default Finetuning Options:")
pprint(sft_trainer.hyperparameters.to_dict()) # rename as hyperparameters

Default Finetuning Options:


{
│   'dataset_max_len': '4096',
│   'global_batch_size': '8',
│   'gradient_clipping': 'True',
│   'gradient_clipping_threshold': '1.0',
│   'learning_rate': '0.0001',
│   'logging_steps': '1',
│   'lora_alpha': '32',
│   'lora_dropout': '0.05',
│   'lora_rank': '16',
│   'lr_scheduler': 'cosine',
│   'lr_warmup_steps_ratio': '0.1',
│   'max_epochs': '5',
│   'merge_weights': 'True',
│   'mlflow_run_id': '',
│   'mlflow_tracking_uri': '',
│   'model_name_or_path': 'meta-llama/Llama-3.2-1B-Instruct',
│   'name': 'example-name-xrs6y',
│   'results_directory': '',
│   'resume_from_path': '',
│   'seed': '42',
│   'temperature': '0.6',
│   'train_val_split_ratio': '0.9',
│   'weight_decay': '0.0'
}

In [8]:
# To update any hyperparameter, simply assign the value, example:
sft_trainer.hyperparameters.global_batch_size=16

#### Start SFT training


In [9]:
training_job = sft_trainer.train(
    wait=True,
)

╭────────────────────────────────── Training Job Status ───────────────────────────────────╮
│  TrainingJob Name      meta-textgeneration-llama-3-2-1b-instruct-sft-20260721002121      │
│  TrainingJob ARN       arn:aws:sagemaker:us-west-2:674622101542:training-job/meta-textg  │
│                        eneration-llama-3-2-1b-instruct-sft-20260721002121                │
│  MLflow Experiment     test-finetuned-models-exp                                         │
│  Links                 ]8;id=4895818;https://us-west-2.console.aws.amazon.com/sagemaker/home?region=us-west-2#/jobs/meta-textgeneration-llama-3-2-1b-instruct-sft-20260721002121\🔗 Training Job (Console)]8;;\                                         │
│                        ]8;id=4895823;https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs$3FlogStreamNameFilter$3Dmeta-textgeneration-llama-3-2-1b-instruct-sft-20260721002121\🔗 CloudWatch Logs]8;;\ | ]8;id=4895824;https://app-3THB6MFIDE4F.mlflow.sagemaker.us-west-2.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IkxWUkRGRyIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNGthWm1uMGlvd2QyWDYzbU9OZG42U2MveStlYVMvOWlRYUtyVXNYQjM0NmdBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVFcmJscFFPVlZNY0hSMk5teFBNRlEyZWxsSVRtUmFSV0pMTm5aMVlrNWlkamRqTnlzMlN6UnVjMGR0V0hSeU56RmhhRk5GWjJoRFkxVnlVSFZuVjBsSFp6MDlBQUVBQjJGM2N5MXJiWE1BUzJGeWJqcGhkM002YTIxek9uVnpMWGRsYzNRdE1qbzVOell4T1RNeU5UUTFOemM2YTJWNUwyVmhNbUZrTUdSa0xXRmtNV0l0TkROalppMDRaak0xTFRNMU5UWmlZVE5pWm1NeE1nQzRBUUlCQUhqRHJwaCtQMVNKcHUzZUl2YnVXUi94bGQ2THMxNjhEU3JZbXo5akp0dlBUQUZIaWtiVUpGRU0zUmxDeDY4YWtIdXBBQUFBZmpCOEJna3Foa2lHOXcwQkJ3YWdiekJ0QWdFQU1HZ0dDU3FHU0liM0RRRUhBVEFlQmdsZ2hrZ0JaUU1FQVM0d0VRUU00YW9OcktqN0lrSHRVWDFOQWdFUWdEdlZ5V1pWdkZYc3V4S09hTml5SDFhd2xNZnpKNGkwTTMrMVNhaXFFYmcwYXRjWWNZUHcwWDBFekZwcWVoQzlFSHdteTlGdjMybFpyNnRMSFFJQUFCQUFyRVVpYk4zcU03Z2pQMVNkRTg0RkJwNFlzUXpZVHgwU1ZkMzJyOWc2RWJsbkh4K0hXUFE4M3VjRWZXTnV1NE5nLy8vLy93QUFBQUVBQUFBQUFBQUFBQUFBQUFFQUFBUVhTTW9zODk5a2NzM3orTkExWXBLYy9tWjV4NU0vRzNjOXBTS3g1Q1ZDbUFQaFloTm5KM0ZnR2FPb3JxMTFsRjR3U3M3emNram1ubG02QzZOY2pnTlp0U3o3WTdwSktUdExOZ1BmcjNoNFVFUUdYSm0wWjUvNGJNeVFsMG1kUlhUR3BJQUdGMmRUZ1M0U2hpVW54ckFEMlZnWTdXUHJ5QWlTL2JLMHkvbWdFU0dCU2d5Rkt2WVRqcFdDbm9sNEMwdjdaczM3eXlmeC9iaURSQlpvTFZmWUgzaEtpcW1QV3YvT295V3lnWU9BOUVFUVlQK1pyaHlqRkkxYlFXVDFxNElBdWt1YThZMCtOZ2xVZU1wNk5mUTcvTkZyMTlZSkZHcEo2cys0MkE2QTVHbkRuY0pkcWNPVjF5Vzl1TlJzS3lzKy81b1pxUlF6S1ZpNjN5QkxHTXhTV0d4eDYrbUpzSEFBWW5talllU01JdVRPMGtTTjdKeFdQV1FXS0lxSUo4Y2tHeTgrN2hBREc1dkttNEo5cElCOHcrYStXdVNvMkY2cUJBK04vQUdGeTZ4MTRhWEl5a0tzOGc2eVByOG0vRm12eU1FRGRuV3M2L3BmcDVIUURjUUU3dmlmdDFyeFdQMUlXRG83UUM5V05DWnZOMjRLcit3NW1ScWp0ejJWT1J6MmRUNnB6S2ZENkFsend1RG5yZk1QcGcrZm9CSUdJTmhiNjdmMy81U3ZMWFk1L0puVjNmaTdBb2RacU5MN3JDb2FSVWdOSVB1SHRZSVc3aWxaaDV0ZkpsNXhEbDNCZGQxMjlZbWJ3RHB5SXU2dUQ1djlFdDhTR1JDWXhlM1NMczUramhNajdVYS9BYTg5ZHNkaE9tRG5mSWNwSkwxV1RkMkhLeFdvR0hrQjRoWUFrL3A4aWxZOXp6TzV5QjhQNzNMNGdBdERla2dVTlpxWnFwa1FxWG9kODhoekRFRzYrcXNab21LWmhoWk43enowcFh0NDFlMFJ6N01vRUVFQ2pHdlMwcmx4cDI4Mk9sc0t0RzVSaUJLS2RoV2xKcHRvQnRZN1Ywc0c5cXhERGtlVVZYb0l1bXAyVjF1bDVwQ0pJTnZLYkh6eHpNTm9qVHdlNzc5cW0yQ1l2Qk51R0diQ3AwaEJ6cjQrZnUrR3NZUWx6Q0Nra0pjWlFjeldDZ3FMNFBOem1jVEtBWGp4UTJzTjVjQlY5MFB6NjFWaTFraENBelNoSXpWS0c4eDcva3VwNk1BNEZ3ZjloaHhscE9aQ3hRQkhmY2ZydkRITjJ4a0tIaGF0OUdFYWR3QVA4a0RIcGVEWllLMHF0OWZCamJKUEVjc0NMVlVMM0JBd3hOeHJOcitrN01kV0VYSHdqYjRGM1NEWFpLeGxQT3pLbnBEZi9pQ2RsYzZwWDlxblVGdW1tZ3gwV2JOZ3dvY09oclRISi9XblR4RFpZNXNiRzlCQS9kc1ExSXRpN1M5bWxVazJZWVhMMzVtekxFdmlqZWNtWjlpTFpML2ltOGUzeFdGdmlVY1V1alFpaWVPanRrN0FtU1dIZHRhUGZ1YVVzaS8raEhPZ1NnanF4a29ZYTVabnZ2ZU16Y1owTC9oMytZdUkrQW5xUkVGTWpCbVFGM2xoOExJMWV6V0Q4dzgxWDFJK1BvQ0RHd1RVRWlCWmYzaXh2VTBVVmxRUjBwZFRKRGk3WVhYdFgxS2pxK1FBdGFkVmhvL2NGZFpyQzRIbHg0K2U3QkNDQWk4SGRWRXVZd1ZwWXlRZEZUUHA4RFhHQ3RleERncytJWkljNDB1S3VYTGNEclZqS0wvU2VZTnVpL29Ya252cGk5YnVrVHd0cUFSYnMwSjJZbFRsNmtpTkxPMlpuajVTZ1N1TmdEcWU3Z0JuTUdVQ01CeFh2dUw5djBJR3lVZ0ZON05SUVMwVmIySTh2MVR4aHpoVjAwbDBGZlRiYn

In [10]:


import json
import re
from sagemaker.core.utils.utils import Unassigned
from sagemaker.core.resources import TrainingJob

response = TrainingJob.get(training_job_name=training_job.training_job_name)

def pretty_print(obj):
    def parse_unassigned(item):
        if isinstance(item, Unassigned):
            return None
        if isinstance(item, dict):
            return {k: parse_unassigned(v) for k, v in item.items() if parse_unassigned(v) is not None}
        if isinstance(item, list):
            return [parse_unassigned(x) for x in item if parse_unassigned(x) is not None]
        if isinstance(item, str) and "Unassigned object" in item:
            pairs = re.findall(r"(\w+)=([^<][^=]*?)(?=\s+\w+=|$)", item)
            result = {k: v.strip("'\"") for k, v in pairs}
            return result if result else None
        return item

    cleaned = parse_unassigned(obj.__dict__ if hasattr(obj, '__dict__') else obj)
    print(json.dumps(cleaned, indent=2, default=str))

pretty_print(response)

{
  "training_job_name": "meta-textgeneration-llama-3-2-1b-instruct-sft-20260721002121",
  "training_job_arn": "arn:aws:sagemaker:us-west-2:674622101542:training-job/meta-textgeneration-llama-3-2-1b-instruct-sft-20260721002121",
  "model_artifacts": "s3_model_artifacts='s3://notebook-test-engine-ds-674622101542-usw2/output/meta-textgeneration-llama-3-2-1b-instruct-sft-20260721002121/output/model'",
  "training_job_status": "Completed",
  "secondary_status": "Completed",
  "hyper_parameters": {
    "dataset_max_len": "4096",
    "global_batch_size": "16",
    "gradient_clipping": "True",
    "gradient_clipping_threshold": "1.0",
    "learning_rate": "0.0001",
    "logging_steps": "1",
    "lora_alpha": "32",
    "lora_dropout": "0.05",
    "lora_rank": "16",
    "lr_scheduler": "cosine",
    "lr_warmup_steps_ratio": "0.1",
    "max_epochs": "5",
    "merge_weights": "True",
    "model_name_or_path": "meta-llama/Llama-3.2-1B-Instruct",
    "name": "example-name-xrs6y",
    "seed": "42",


In [11]:
#In order to skip waiting and monitor the training Job later

'''
training_job = sft_trainer.train(
    wait=False,
)
'''

'\ntraining_job = sft_trainer.train(\n    wait=False,\n)\n'

In [12]:
pretty_print(training_job)

{
  "training_job_name": "meta-textgeneration-llama-3-2-1b-instruct-sft-20260721002121",
  "training_job_arn": "arn:aws:sagemaker:us-west-2:674622101542:training-job/meta-textgeneration-llama-3-2-1b-instruct-sft-20260721002121",
  "model_artifacts": "s3_model_artifacts='s3://notebook-test-engine-ds-674622101542-usw2/output/meta-textgeneration-llama-3-2-1b-instruct-sft-20260721002121/output/model'",
  "training_job_status": "Completed",
  "secondary_status": "Completed",
  "hyper_parameters": {
    "dataset_max_len": "4096",
    "global_batch_size": "16",
    "gradient_clipping": "True",
    "gradient_clipping_threshold": "1.0",
    "learning_rate": "0.0001",
    "logging_steps": "1",
    "lora_alpha": "32",
    "lora_dropout": "0.05",
    "lora_rank": "16",
    "lr_scheduler": "cosine",
    "lr_warmup_steps_ratio": "0.1",
    "max_epochs": "5",
    "merge_weights": "True",
    "model_name_or_path": "meta-llama/Llama-3.2-1B-Instruct",
    "name": "example-name-xrs6y",
    "seed": "42",


### View any Training job details

We can get any training job details and its status with TrainingJob.get(...)

In [13]:
from sagemaker.core.resources import TrainingJob

response = TrainingJob.get(training_job_name=training_job.training_job_name)
pretty_print(response)

{
  "training_job_name": "meta-textgeneration-llama-3-2-1b-instruct-sft-20260721002121",
  "training_job_arn": "arn:aws:sagemaker:us-west-2:674622101542:training-job/meta-textgeneration-llama-3-2-1b-instruct-sft-20260721002121",
  "model_artifacts": "s3_model_artifacts='s3://notebook-test-engine-ds-674622101542-usw2/output/meta-textgeneration-llama-3-2-1b-instruct-sft-20260721002121/output/model'",
  "training_job_status": "Completed",
  "secondary_status": "Completed",
  "hyper_parameters": {
    "dataset_max_len": "4096",
    "global_batch_size": "16",
    "gradient_clipping": "True",
    "gradient_clipping_threshold": "1.0",
    "learning_rate": "0.0001",
    "logging_steps": "1",
    "lora_alpha": "32",
    "lora_dropout": "0.05",
    "lora_rank": "16",
    "lr_scheduler": "cosine",
    "lr_warmup_steps_ratio": "0.1",
    "max_epochs": "5",
    "merge_weights": "True",
    "model_name_or_path": "meta-llama/Llama-3.2-1B-Instruct",
    "name": "example-name-xrs6y",
    "seed": "42",


## Continued Finetuning (or) Finetuning on Model Artifacts

#### Fetch the Model Package ARN from a previous Training Job

After a serverless training job completes, it produces an `OutputModelPackageArn` that can be used as the base model for iterative (continued) fine-tuning. Here's how to retrieve it:

In [14]:
from sagemaker.core.resources import TrainingJob

# Get the completed training job
previous_job = TrainingJob.get(training_job_name=training_job.training_job_name)

# The output model package ARN is available on completed serverless training jobs
output_model_package_arn = previous_job.output_model_package_arn
print(f"Output Model Package ARN: {output_model_package_arn}")

# Use this ARN to get the ModelPackage for iterative training
from sagemaker.core.resources import ModelPackage

model_package = ModelPackage.get(model_package_name=output_model_package_arn)
pretty_print(model_package)

Output Model Package ARN: arn:aws:sagemaker:us-west-2:674622101542:model-package/test-model-package-group/7


{
  "model_package_group_name": "test-model-package-group",
  "model_package_version": 7,
  "model_package_registration_type": "Logged",
  "model_package_arn": "arn:aws:sagemaker:us-west-2:674622101542:model-package/test-model-package-group/7",
  "creation_time": "2026-07-21 00:31:42.727000+00:00",
  "inference_specification": "containers=[ContainersItem(model_data_url=None, image=None, nearest_model_name=None, model_data_source=ModelDataSource(s3_data_source=S3ModelDataSource(s3_data_type='S3Prefix', compression_type='None', s3_uri='s3://notebook-test-engine-ds-674622101542-usw2/output/meta-textgeneration-llama-3-2-1b-instruct-sft-20260721002121/output/model/', model_access_config=Unassigned(), hub_access_config=Unassigned(), manifest_s3_uri=Unassigned(), e_tag=Unassigned(), manifest_etag=Unassigned())), is_checkpoint=False, base_model=BaseModel(hub_content_name='meta-textgeneration-llama-3-2-1b-instruct', hub_content_version='2.9.0', recipe_name='llmft_llama3_2_1b_instruct_seq4k_gpu_

#### Discover a ModelPackage and get its details

In [15]:
from rich import print as rprint
from rich.pretty import pprint
from sagemaker.core.resources import ModelPackage, ModelPackageGroup

#model_package_iter = ModelPackage.get_all(model_package_group_name="test-finetuned-models-gamma")
model_package = ModelPackage.get(model_package_name=output_model_package_arn)

pretty_print(model_package)

{
  "model_package_group_name": "test-model-package-group",
  "model_package_version": 7,
  "model_package_registration_type": "Logged",
  "model_package_arn": "arn:aws:sagemaker:us-west-2:674622101542:model-package/test-model-package-group/7",
  "creation_time": "2026-07-21 00:31:42.727000+00:00",
  "inference_specification": "containers=[ContainersItem(model_data_url=None, image=None, nearest_model_name=None, model_data_source=ModelDataSource(s3_data_source=S3ModelDataSource(s3_data_type='S3Prefix', compression_type='None', s3_uri='s3://notebook-test-engine-ds-674622101542-usw2/output/meta-textgeneration-llama-3-2-1b-instruct-sft-20260721002121/output/model/', model_access_config=Unassigned(), hub_access_config=Unassigned(), manifest_s3_uri=Unassigned(), e_tag=Unassigned(), manifest_etag=Unassigned())), is_checkpoint=False, base_model=BaseModel(hub_content_name='meta-textgeneration-llama-3-2-1b-instruct', hub_content_version='2.9.0', recipe_name='llmft_llama3_2_1b_instruct_seq4k_gpu_

#### Create Trainer

Trainer creation is same as above Finetuning Section except for `model`'s input is ModelPackage(previously trained artifacts)

In [16]:
from sagemaker.core.resources import ModelPackageGroup
# For fine-tuning 
sft_trainer = SFTTrainer(accept_eula=True, 
    model=model_package, # Union[str, ModelPackage]
    training_type=TrainingType.LORA, 
    model_package_group=ModelPackageGroup.create(model_package_group_name="sdk-test-finetuned-models"), # Make it Optional
    mlflow_experiment_name="test-finetuned-models-exp", # Optional[str]
    mlflow_run_name="test-finetuned-models-run", # Optional[str]
    training_dataset=dataset.arn, #Optional[]
    s3_output_path="s3://notebook-test-engine-ds-674622101542-usw2/output/",  # TODO: replace with your S3 output path
)


[07/21/26 00:31:47] INFO     Creating model_package_group resource.                              ]8;id=4895829;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4895830;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#23250\23250]8;;\

[07/21/26 00:31:48] INFO     SageMaker session not provided. Using default Session.                  ]8;id=4895835;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=4895836;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

#### Start the Training

In [17]:
training_job = sft_trainer.train(
    wait=True,
)

╭────────────────────────────────── Training Job Status ───────────────────────────────────╮
│  TrainingJob Name      meta-textgeneration-llama-3-2-1b-instruct-sft-20260721003150      │
│  TrainingJob ARN       arn:aws:sagemaker:us-west-2:674622101542:training-job/meta-textg  │
│                        eneration-llama-3-2-1b-instruct-sft-20260721003150                │
│  MLflow Experiment     test-finetuned-models-exp                                         │
│  Links                 ]8;id=4899790;https://us-west-2.console.aws.amazon.com/sagemaker/home?region=us-west-2#/jobs/meta-textgeneration-llama-3-2-1b-instruct-sft-20260721003150\🔗 Training Job (Console)]8;;\                                         │
│                        ]8;id=4899795;https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs$3FlogStreamNameFilter$3Dmeta-textgeneration-llama-3-2-1b-instruct-sft-20260721003150\🔗 CloudWatch Logs]8;;\ | ]8;id=4899796;https://app-3THB6MFIDE4F.mlflow.sagemaker.us-west-2.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IkdDSldaNSIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNElvYllhREx0L3l6VnZUVGlNMnZ2azdqZTUzenJqVTlraldHMDhscHN5S1VBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVFelUwYzJRV1pFZFUxVk9IUlZWbEpXUjFFMlZuSXhlRzlvVkZscFUzUnhkbVY2VEc5TFdXaGhPRkl5VjBJdlNteHVTRUpsWkZSSFVXczRiRkYwWW1kM2R6MDlBQUVBQjJGM2N5MXJiWE1BUzJGeWJqcGhkM002YTIxek9uVnpMWGRsYzNRdE1qbzVOell4T1RNeU5UUTFOemM2YTJWNUwyVmhNbUZrTUdSa0xXRmtNV0l0TkROalppMDRaak0xTFRNMU5UWmlZVE5pWm1NeE1nQzRBUUlCQUhqRHJwaCtQMVNKcHUzZUl2YnVXUi94bGQ2THMxNjhEU3JZbXo5akp0dlBUQUhMRS9pK3NFTjJFdC85cE8xU25IcHJBQUFBZmpCOEJna3Foa2lHOXcwQkJ3YWdiekJ0QWdFQU1HZ0dDU3FHU0liM0RRRUhBVEFlQmdsZ2hrZ0JaUU1FQVM0d0VRUU0rSnRZcjlXK0d1bjhvbW1oQWdFUWdEc2FWT2M4N21EcSsvNG9vdnkxZGJTb2UrNnIwMnBVWUgyY2htd3RRcEx0cTZiUlM0QU81cStnYW9pRVovdEJiVGluVDhYdnBrT2NVNkc3endJQUFCQUFXSklCU05raWMyMXFnQlA5WWtBZUhUbFJjeFEwTHp0QW5UcmVwT1h4cnpPbFBvTlFHMGlyTWhpc0RHQURrQUtRLy8vLy93QUFBQUVBQUFBQUFBQUFBQUFBQUFFQUFBUVhiYXhsWFpvRUwwYkZrcmNNbG5lVUV5TGxTamk1ZTdIUVZPK0srMnpCU3F5ekQyQUNDVzQvcUUzZWtySTFQVGtGMzdJQ1VtU2Q5L1daOWh3eW9zL3YwNEYyNGlBcmp0SmFobWhCcWFWbmw5RnJOeTdVUy9icm53T01GVnAwZW14RmdUNHVHZTV0R0NTNUlSa2NTTmxFOWY3SXVZcTZZNHIzU0VKbDExUGlvdWtJSFQxY1M4b2pTWWR5MWZHSnp4TlE4YUdxQjZyZ0tuakF1eGV4NzRPamZ5MnF4ZGYySHQ0U3E3RDEvbG8yeWtnZUdSOENCYVY1NjZhdFlvWDcxTHFMaXNoenBpR1JRNjFHdWdPdStNWFk0enM1SnpiUThUQkxxNjNIUDhCbHpzYzhUalBjTkF6MVhaYUgzK3NhRWM0alNuV0diMXlsazZnSHI1Y1BoYndwcGRYL2FMcVF2VjBRU09mS09kbTZUUlM5bVJDaWJ0ZmFZZjl6SzlFckZ6cStYcVNQcXkrSVVVYmF4bFNLeWRLZ3dZS2xOL1YxRUlQV1FsSkdnbHBQNXVTU2NVU29sUytwWXZ4cFM4ekFnTmZJenpHVWRvN0dOMmRpeW1oRWhETWJNMVAxYWowaXVCc3V3SmhnV1ByVTNkZUYwa3F0d1ZQamRmQTNDZ09EcStnczVNOFJnRXQ3RDFRWE01clJVb1B4M2JkV2craDhFSFNQS3VoelVEdE9WRGcrbzQxY3l4RkRJVnViVllaRzJRbWIrWVk0dGxseEtwZHFpb2UyMnZheVFUYW5nNHJIQXE2Tkw2VFVUank3ejNlb1hHQXo1dndMOVNudXpyN2lxeTl0V3VSRnkvQU4xT1dkRHhDWE9mZ2IwUUlsRy9OT0NnaEFxd01LK3RGOXU2dDl4Qjg5Y0Zianh0MlBDaHpmSWlPQnNwOFNMb0tvaGhkM016ZHZORmxlUzdBUTdTQ0RaV3pPTm51YytrWmRNN1lINjJ1VlA1ZDdtaGNEaU1Sc3R1NkJnQ3V4bHgzZFRtcXcyMG5mK0VjYVBvNk54ZE52YVZGZHJUYUhqcG1XUG5TamFrTmtqdFdONG5jN0k5dFh0dEFQMjlWVzFGVUoxSFJNYy9xRFdRUXVFU1B1bGFINVF5MGtDaC9ieU5Ka1puTGdla1BqSEpzWFVtcHBHWEd1Qk1sL0M1REsvU3hyWnVCYXE4ZE16MjY0azBHZ2dHb2tZcDdsY2xqQUhrcnNKK0wxSHhJTjNGeUNCMDBLUmR2Y2FsUFpVV0hndWZwQXk5aXFDK1JtT3hxTVVyeTNXL3NTdjU1b1JFK3JpUXAyWVZ0MUNhWmNkN2hoNmVpUUV6dnIzM3B2Q2RrcmNzOVJ6dWcvYmh5U0s5aDI4RWhSeTRySlRTaDRMODlSZVE1cHdrS3FpVWlPMG9ORlpjdXNUR1pieVkwUi83L1ZzVnJWejRQZ0c1TUpNMDEvanBSWGhuektZMHVOSkpVSlJIY1d5b2E1NWhGUmhJSEE4RGZSYU92UEhydFJjam9kS2dIbjM1NVdGK3RmSkZoei9QdGp3dVFTR0RWdG5PTkw0MFFuRnNCajl2dzlNbmRKQ0Q5R1hIamNsVUQvQ2hzdU9lZExSd2hRUWxtOGFqZkROT0E2UGJJZXFwenBiK08wZHdxZ0tuVEcyMEx6SHFKTjJDZ2xabjM2aFUxZ2pvMXJHMVo4T24yeVpCYzRZNUN2R1pBODV6bG5xQ0ZjbzBNS1dJVmVZd3A3MVF3TmVpQUxkK09MNVR2cUZOQXp6OFkrMkpUUUMxTWZaNkZvaDdhcEZrMk9CVC9Hc3E2OUhMNVhBbzQxejN1aC84UGx4elRLZVlTVmo1TkEyQTRjMUZReTFCbkZlZ0JuTUdVQ01RRHlTekFGQXkvaWhBa2NudDBSb25Ub043T3Fja1Q4d0ptcitoUFVuMjJEWl

In [18]:
pretty_print(training_job)

{
  "training_job_name": "meta-textgeneration-llama-3-2-1b-instruct-sft-20260721003150",
  "training_job_arn": "arn:aws:sagemaker:us-west-2:674622101542:training-job/meta-textgeneration-llama-3-2-1b-instruct-sft-20260721003150",
  "model_artifacts": "s3_model_artifacts='s3://notebook-test-engine-ds-674622101542-usw2/output/meta-textgeneration-llama-3-2-1b-instruct-sft-20260721003150/output/model'",
  "training_job_status": "Completed",
  "secondary_status": "Completed",
  "hyper_parameters": {
    "dataset_max_len": "4096",
    "global_batch_size": "8",
    "gradient_clipping": "True",
    "gradient_clipping_threshold": "1.0",
    "learning_rate": "0.0001",
    "logging_steps": "1",
    "lora_alpha": "32",
    "lora_dropout": "0.05",
    "lora_rank": "16",
    "lr_scheduler": "cosine",
    "lr_warmup_steps_ratio": "0.1",
    "max_epochs": "5",
    "merge_weights": "True",
    "model_name_or_path": "meta-llama/Llama-3.2-1B-Instruct",
    "name": "example-name-xrs6y",
    "seed": "42",
 

## Finetuning with Serverful Compute (TrainingJobCompute)

Use `TrainingJobCompute` to run training on dedicated instances. This enables:
- Recipe overrides and validation
- Iterative training from S3 checkpoints via `base_model_name`
- Uncompressed output with `disable_output_compression=True`

In [19]:
from sagemaker.core.training.configs import TrainingJobCompute

# Base training on serverful compute
sft_trainer_serverful = SFTTrainer(
    model="meta-textgeneration-llama-3-2-1b-instruct",
    training_type=TrainingType.LORA,
    compute=TrainingJobCompute(instance_type="ml.g5.xlarge", instance_count=1),
    training_dataset="s3://notebook-test-engine-ds-674622101542-usw2/datasets/sample-sft-dataset.jsonl",
    s3_output_path="s3://notebook-test-engine-ds-674622101542-usw2/output/",
    disable_output_compression=True,
    accept_eula=True,
)

training_job = sft_trainer_serverful.train(wait=False)
print(f"Serverful SFT job submitted: {training_job.training_job_name}")

[07/21/26 00:45:56] INFO     SageMaker session not provided. Using default Session.                  ]8;id=4899801;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=4899802;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

[07/21/26 00:45:57] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=4899807;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=4899808;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     SageMaker session not provided. Using default Session.                  ]8;id=4899813;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=4899814;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

[07/21/26 00:45:58] INFO     Cannot simulate policies for                                  ]8;id=4899819;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=4899820;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=4899825;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=4899826;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

...

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=4899899;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=4899900;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#167\167]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     Training image URI:                                               ]8;id=4899906;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=4899907;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             920498770698.dkr.ecr.us-west-2.amazonaws.com/hyperpod-recipes:llm                     
                             ft-v1.0.0                                                                             

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=4899912;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=4899913;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/21/26 00:46:02] INFO     Creating training_job resource.                                     ]8;id=4899918;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4899919;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31239\31239]8;;\

Serverful SFT job submitted: meta-textgeneration-llama-3-2-1b-instruct-SFT-20260721004601


In [20]:
# Iterative training from S3 checkpoint (serverful)
# Requires base_model_name to identify model for recipe lookup
sft_trainer_iterative = SFTTrainer(
    model="s3://notebook-test-engine-ds-674622101542-usw2/sft-output/meta-textgeneration-llama-3-2-1b-instruct-sft-20260716170815/output/model/checkpoints/hf_merged/",
    base_model_name="meta-textgeneration-llama-3-2-1b-instruct",
    training_type=TrainingType.LORA,
    compute=TrainingJobCompute(instance_type="ml.g5.xlarge", instance_count=1),
    training_dataset="s3://notebook-test-engine-ds-674622101542-usw2/datasets/sample-sft-dataset.jsonl",
    s3_output_path="s3://notebook-test-engine-ds-674622101542-usw2/iterative-output/",
    disable_output_compression=True,
    accept_eula=True,
)

training_job = sft_trainer_iterative.train(wait=False)
print(f"Iterative SFT job submitted: {training_job.training_job_name}")

[07/21/26 00:46:03] INFO     SageMaker session not provided. Using default Session.                  ]8;id=4899924;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=4899925;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

[07/21/26 00:46:04] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=4899930;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=4899931;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     SageMaker session not provided. Using default Session.                  ]8;id=4899936;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=4899937;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

[07/21/26 00:46:05] INFO     Cannot simulate policies for                                  ]8;id=4899942;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=4899943;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=4899948;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=4899949;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

...

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=4900014;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=4900015;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#167\167]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     Training image URI:                                               ]8;id=4900020;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=4900021;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             920498770698.dkr.ecr.us-west-2.amazonaws.com/hyperpod-recipes:llm                     
                             ft-v1.0.0                                                                             

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=4900026;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=4900027;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/21/26 00:46:09] INFO     Creating training_job resource.                                     ]8;id=4900032;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4900033;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31239\31239]8;;\

Iterative SFT job submitted: meta-textgeneration-llama-3-2-1b-instruct-SFT-20260721004608


#### SFT Trainer Nova testing

In [21]:
from sagemaker.core.resources import ModelPackageGroup
os.environ['SAGEMAKER_REGION'] = 'us-west-2'

# For fine-tuning 
sft_trainer_nova = SFTTrainer(accept_eula=True, 
    #model="test-nova-lite-v2", 
    #model="nova-textgeneration-micro",
    model="nova-textgeneration-micro",
    training_type=TrainingType.LORA, 
    model_package_group=ModelPackageGroup.create(model_package_group_name="sdk-test-finetuned-models"), 
    mlflow_experiment_name="test-nova-finetuned-models-exp", 
    mlflow_run_name="test-nova-finetuned-models-run", 
    training_dataset="arn:aws:sagemaker:us-west-2:674622101542:hub-content/R8VBAD7SJ9M4M5ATDE0MIPSEJU3M1TEH5UCUNB65JMGNEP87RHFG/DataSet/nte-sft-dataset/1.0.0",
    s3_output_path="s3://notebook-test-engine-ds-674622101542-usw2/output/"  # TODO: replace with your S3 output path
)


[07/21/26 00:46:10] INFO     Creating model_package_group resource.                              ]8;id=4900038;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4900039;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#23250\23250]8;;\

                    INFO     SageMaker session not provided. Using default Session.                  ]8;id=4900044;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=4900045;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

In [22]:
sft_trainer_nova.hyperparameters.to_dict()

{'name': 'my-lora-run-fd8as',
 'global_batch_size': '64',
 'max_epochs': '2',
 'learning_rate': '1e-05',
 'lora_alpha': '128',
 'learning_rate_ratio': '16.0',
 'max_context_length': '8192',
 'weight_decay': '0.0',
 'warmup_steps': '10',
 'min_lr': '1e-06'}

In [23]:
training_job = sft_trainer_nova.train(
    wait=True,
)

╭────────────────────────────────── Training Job Status ───────────────────────────────────╮
│  TrainingJob Name      nova-textgeneration-micro-sft-20260721004612                      │
│  TrainingJob ARN       arn:aws:sagemaker:us-west-2:674622101542:training-job/nova-textg  │
│                        eneration-micro-sft-20260721004612                                │
│  MLflow Experiment     test-nova-finetuned-models-exp                                    │
│  Links                 ]8;id=4908970;https://us-west-2.console.aws.amazon.com/sagemaker/home?region=us-west-2#/jobs/nova-textgeneration-micro-sft-20260721004612\🔗 Training Job (Console)]8;;\                                         │
│                        ]8;id=4908975;https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs$3FlogStreamNameFilter$3Dnova-textgeneration-micro-sft-20260721004612\🔗 CloudWatch Logs]8;;\ | ]8;id=4908976;https://app-3THB6MFIDE4F.mlflow.sagemaker.us-west-2.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IklGM0FJNiIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNENuc1JMOWNQcUN5NmtPK0JhbkZWbEFPMlpRNTB4Z3M4SSs4aW5jSjZWWHdBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVFMFREbFlLMFJ5U1VnMGIxSm5NVkJQSzB4NFpuSktTRWN6VkdsNVVHVkZPR0UwZEM5bVFXRnJPWHBxZVVOTGVtbGxRVzV3Y0U1eFR6VnNUbk5ZZW5GRWR6MDlBQUVBQjJGM2N5MXJiWE1BUzJGeWJqcGhkM002YTIxek9uVnpMWGRsYzNRdE1qbzVOell4T1RNeU5UUTFOemM2YTJWNUwyVmhNbUZrTUdSa0xXRmtNV0l0TkROalppMDRaak0xTFRNMU5UWmlZVE5pWm1NeE1nQzRBUUlCQUhqRHJwaCtQMVNKcHUzZUl2YnVXUi94bGQ2THMxNjhEU3JZbXo5akp0dlBUQUdRTjlub0k5eVMrcUR3ejUxZnlDRU5BQUFBZmpCOEJna3Foa2lHOXcwQkJ3YWdiekJ0QWdFQU1HZ0dDU3FHU0liM0RRRUhBVEFlQmdsZ2hrZ0JaUU1FQVM0d0VRUU1oWXJpVVBEZXE5UFFoTklBQWdFUWdEdm5BV1ZVL3NGNFRCVldkd3NFelZPNGVybGFqVldDWkg1ZkY0M3drYXE5SkFOQ1pXYVp3S1VUMlh2THJocklheFNUV1dJbHArVyt6MEVqaGdJQUFCQUFNQUFRUkJtNWpPZWpNZk0rOWlhU2haeXJuRVdwZFFTRStRZDNhbWpaNE5XUzRyaDVoYVNWOGhLOHJiYTIwcXEvLy8vLy93QUFBQUVBQUFBQUFBQUFBQUFBQUFFQUFBUVgwb2R1bnBMdXpweHFJQmVwaVRZNTB4NE9XTkVtaWhISHdhdnZwRzloVDIvUzl6NU90UFpHL2Q3OG81TStZQUJjN3BHTmozblJhVU12ZlhLZ2oxWWRLMncvMkhRSVZ6NVZGaHFJRnlhM2FWcU54amdwS04xanVZVlExcWxGa0g3RG9EbkF1VWJOeWZjY01UZjZxUWZkc0RWMTdFVG5LcjRjSXpmNXl0ZDdPVlpXNEl4Q3JqWXRuZXRPOXpmREY4aXIrY3RMcHFFcXlPbXNxQUwva2tXbXlEQzY1L3dRRmNQazk1UFdIYmNibXN1aVJGdTRwcWtOaGkwclNNS2drc3ZuZnY5WDZzQVBBeGhPRkFFRkgybEZJSUtTRUxNNVN4a2hJQjd0b3JPU3AvVk4wYW1Cc3BTMDZ1QVJTamFhUGluMmV5RnhGa2lzaktYVHV5YlpHTkh5VmZpclYwcGtGckdYS2hKYUFKTFFNSzR0d2drYTVnajVYS0ZGMFJ2WXpYeG9uU3RIaTZJYTNHZjNWTFVZekI1akUzNGhpcHIxOXBCUVhUZEliOTZpUkxsUXhPR1lEdzBlQ2FRLytxK0E2dm1kaEhQb05nMitxT01uVnhLQkJKS09iL2VPU2lRUTlYbG9kUFJVZTV4b0ZhYy9jNFNnT3JJeWNKUy9GV3owclZ1R0pvUXFtcVpNdllmSDRpazlaWGtlQmhWdjNodnFpdy9QT2JibUVnOXV5Wkh2VHZLMjQ4T1MxbE12N1I5Y0JaTFVkbkZKR2QrSEdOblNRUmVmODFYd0xONE5oSkw0MmZwbTB0WDZmc3FDd1UwUFB4TFRMZkhMRm9zVDVaZWRLZVdQUFRzd09BNStwejgrVEY0WVJLMkN5ZklxbXlNbWhRSFVVUFVHOFBIV3RSV3VqMkFmWkNHK1ltT0ZRbVhrNXRSQ2tGc1FkR2J1QUZLTDNPYWIrd2VKaElORS9GVHdZOGZKUGxSeFdFYmdQZ2w5TGhoR2dvMXBrL3FVc0RiRDZBN25KWFhLL2ZPMGpYajQ3VG5YWVk1TFc4Y004SGdxSkt4dklUWGlnclozcTJsVjVGV0d1YUMzVkdhL2wxS2dwakViczBHUFN1STI0aHJSWkgwMkNTa2R6Yk5DMW5ZRzRpcU9jUmVqSlQ5cmxEdlBDQTN5YVB5N2s5OTdmcmIxVjZzam9idm1kR0NIeERBTngvMHowaHVCcXFCT3BaTlBiaC9FUkZXNUxGWjhxVEd2bVB3WUhsNnhNcTBSR0swVnc2Tm1JQWI1YUpVY1BLdm5uYzZlVGtLNEJjdXNEY0JoQ2NPUU05MEE5S1ZmUitoRjU3SlNRR24ybSt2VzBPcTJid0NwM1c0R2V5S0xmU21RZlNMaFQ4d0R4eXUrR2ZETGtJQW43WURwbzdlVUZGZ0tlNVlEWjhZVVdRY1huRVZoR0I2MUNtUHh0ZnFKa0I2YnJKVFZWeU1kS05zK1ltNldrcVA3anZtbUJVRjd0blM2RE53TGE2SzhJaVAvZ2hMUzNaSmhQWlNaZnVRaVBnOGVnVmd2Rk1kNUZPQWtmeW9KUHQ2aENhZVc2QlFoZlpLNitKbDMvN2lnREtwelBneU1FbkJ0Y2lwVDRsOTlHcVN6WGUwRVJOV09LRkhXbUg4ZUtJWVJ6VEtqN3NTSEZtV2tSK29MTFZzbXhIKzNpSmo2NVEzblpwUlBxclEzNDdpUUlGNDFLS0hpSTQ3cEVicXl3OE1HZlVhVEtiVU5IZVhUOERPdGt5MDV3djZTZGI0MWs5YjdsSEU4Sno2YmhOQnFvbkJMU2NxcmNOS2NpVHNRQktvTzNNcnZOdS9tQVk1NDZkUTZWeFZaa3laeGdjMkNURllpWlI4NjhIK1YzZ0JuTUdVQ01RRFQ1d29IbFFXTDJvZ25pTkE1TVQyTDlmbHRUTFcyUjgxK2tzWmRkdHNPckVaeTFUWHJwVklVYzl5SjlXSzgxQVFDTU